In [13]:
from src.data_loader import load_training_data
from src.preprocessing import notch_filter, bandpass_filter, highpass_filter
import config
import numpy as np
import scipy
from math import log, e
import time

In [2]:
import pywt

Functions

In [3]:
def entropy2(labels, base=None):
  """ Computes entropy of label distribution. """

  n_labels = len(labels)

  if n_labels <= 1:
    return 0

  value,counts = np.unique(labels, return_counts=True)
  probs = counts / n_labels
  n_classes = np.count_nonzero(probs)

  if n_classes <= 1:
    return 0

  ent = 0.

  # Compute entropy
  base = e if base is None else base
  for i in probs:
    ent -= i * log(i, base)

  return ent

In [5]:
def wavelet_feature_extraction(coeff_arr):
    """Function to extract signals from wavelet coefficients from 
    the decomposition
    
    Args:
        coeff_arr (np.ndarray): 2D array of all coefficients from array
    
    Returns:
        wavelet_features (dict): A dictionary of all wavelet features"""

    wavelet_features = {}
    # Try extracting the entropy, and other statistics from every coefficient if possible
    for coeff in coeff_arr:
        # Do the feature extraction here for every coeff
        energy = np.sum(coeff**2)
        wavelet_features["energy"] = energy
    
        # Get statistical moments
        # Mean
        mean = np.mean(coeff)
        wavelet_features["mean"] = mean

        # Standard Deviation
        stdev = np.std(coeff)
        wavelet_features["stdev"] = stdev

        # Skewness
        skewness = scipy.stats.skew(coeff) 
        wavelet_features["skewness"] = skewness

        # Kurtosis
        kurt = scipy.stats.kurtosis(coeff)
        wavelet_features["kurtosis"] = kurt

        # Entropy
        entropy = entropy2(coeff)
        wavelet_features["entropy"] = entropy
    
    return(wavelet_features)







In [6]:
def wavelet_decomposition(signal: np.ndarray, wavelet_name: str):
    """Function to do wavelet decomposition on a signal
    
    Args:
        signal (np.ndarray): 1D array of a signal for wavelet decomposition
        wavelet_name (str): name of wavelet family and the number e.g 'coif1' or 'db1'

    Returns:
        coeff_arr (np.ndarray): Array of coefficients of different levels
    """
    # Get wavelet 
    wavelet = pywt.Wavelet(wavelet_name)
    # Do wavelet decomposition
    coeff_arr = pywt.wavedec(signal, wavelet)
    
    return(coeff_arr)



    

In [7]:
def wavelet_processing(epoch, wavelet_name):
    coeff_arr = wavelet_decomposition(signal = epoch, wavelet_name = wavelet_name)
    wavelet_features = wavelet_feature_extraction(coeff_arr=coeff_arr)
    return(wavelet_features)



In [8]:
edf_file = "../data/training/R1.edf"
xml_file = "../data/training/R1.xml"
multi_channel_data, labels, info = load_training_data(
            edf_file, xml_file
        )

Loading training data from ../data/training/R1.edf and ../data/training/R1.xml...


c:\Users\Ines\Documents\2025\KTH\Studies\Signal Processing and Data Analysis\Project\CM2013\Python\src\data_loader.py:56: RuntimeWarning: Invalid measurement date encountered in the header.
  raw = mne.io.read_raw_edf(edf_file_path, preload=True, verbose=False)


Identified channels:
  EEG: ['EEG(sec)', 'EEG']
  EOG: ['EOG(L)', 'EOG(R)']
  EMG: ['EMG']
  EEG: 2 channels, 3750 samples/epoch, 125.0 Hz
  EOG: 2 channels, 3750 samples/epoch, 125.0 Hz
  EMG: 1 channels, 3750 samples/epoch, 125.0 Hz

Loaded 1083 epochs (9.03 hours)
Sleep stage distribution:
  Wake: 332 epochs (30.7%)
  N1: 47 epochs (4.3%)
  N2: 457 epochs (42.2%)
  N3: 145 epochs (13.4%)
  REM: 102 epochs (9.4%)


In [16]:
multi_channel_data["eeg"].shape

(1083, 2, 3750)

In [9]:
# Extracting single signal
signal = multi_channel_data["eeg"][0][1]

In [10]:
len(signal) # The signal is one epoch, which we will be extracting the spectral features from

3750

Try Functions for Processing

In [38]:
start = time.time()
wavelet_features = wavelet_processing(signal, "coif1")
end = time.time()
print(f"Elapsed time: {end - start}")
print(f"Estimated time for processing of the whole signal: {1083*(end - start)} seconds")

Elapsed time: 0.013338565826416016
Estimated time for processing of the whole signal: 14.445666790008545 seconds


In [39]:
wavelet_features

{'energy': np.float64(6.361007292329731e-07),
 'mean': np.float64(-9.479026668592219e-09),
 'stdev': np.float64(1.840902353066617e-05),
 'skewness': np.float64(0.1016465527453623),
 'kurtosis': np.float64(1.998854542298929),
 'entropy': np.float64(7.535214329029978)}

Check Wavelets

In [22]:
pywt.families()
pywt.wavelist('coif')



['coif1',
 'coif2',
 'coif3',
 'coif4',
 'coif5',
 'coif6',
 'coif7',
 'coif8',
 'coif9',
 'coif10',
 'coif11',
 'coif12',
 'coif13',
 'coif14',
 'coif15',
 'coif16',
 'coif17']

Testing PyWavelet module

In [32]:
# Daubechies coefficient
db1 = pywt.Wavelet('db1')

In [33]:
c = pywt.wavedec(signal, db1)

In [34]:
c

[array([ 1.6247858e-06, -1.6384340e-04]),
 array([7.48051383e-05, 1.30611115e-04]),
 array([ 5.24509804e-05, -3.77757353e-05,  4.50367647e-06,  2.29166667e-04]),
 array([-1.06152672e-04, -1.40814770e-04,  4.28943452e-06, -3.03293350e-05,
        -7.53034060e-05, -1.42547874e-05,  1.69406589e-21,  0.00000000e+00]),
 array([ 4.92647059e-05, -1.49142157e-04, -7.75735294e-05, -1.11029412e-04,
         2.26654412e-04, -1.06250000e-04,  3.41911765e-05, -3.16176471e-05,
         2.16911765e-05,  8.48039216e-05, -1.24387255e-05,  1.00490196e-05,
        -1.50122549e-05, -2.98406863e-05,  1.45036765e-04]),
 array([-2.56499519e-05, -1.47313913e-05,  2.19584385e-04, -1.35008868e-04,
         1.46880637e-04,  5.25997324e-05, -1.89774982e-05,  1.18717683e-05,
         1.20450787e-05,  8.75217952e-05, -3.37088894e-05, -1.02513152e-04,
         7.79897185e-07,  2.85962301e-06, -5.54593554e-05,  9.01214525e-06,
        -3.12825426e-05,  3.89082040e-05,  4.30676557e-05,  6.69845027e-05,
         7.9722

In [35]:
num_of_coeffs = len(c)
num_of_coeffs

12

In [36]:
# Number of levels in the decomposition
pywt.dwt_max_level(len(signal), db1)

11

In [37]:
# Try extracting the entropy, and other statistics from every coefficient if possible
for coeff in c:
    # Do the feature extraction here for every coeff
    energy = np.sum(coeff**2)
    
    print(f"Energy: {energy}")

    # Get statistical moments
    # Mean
    mean = np.mean(coeff)
    print(f"Mean: {mean}")

    # Standard Deviation
    stdev = np.std(coeff)
    print(f"Standard deviation: {stdev}")

    # Skewness
    skewness = scipy.stats.skew(coeff) 
    print(f"Skewness: {skewness}")

    # Kurtosis
    kurt = scipy.stats.kurtosis(coeff)
    print(f"Kurtosis: {kurt}")

    # Entropy
    entropy = entropy2(coeff)
    print(f"Entropy: {entropy}")







Energy: 2.6847299754000974e-08
Mean: -8.110930725375133e-05
Standard deviation: 8.273409305610988e-05
Skewness: 0.0
Kurtosis: -2.0
Entropy: 0.6931471805599453
Energy: 2.2655072005058196e-08
Mean: 0.00010270812651977406
Standard deviation: 2.7902988179175055e-05
Skewness: -7.988605391462577e-16
Kurtosis: -2.0
Entropy: 0.6931471805599453
Energy: 5.671575573397258e-08
Mean: 6.208639705882355e-05
Standard deviation: 0.000101608160271443
Skewness: 0.8285044360233955
Kurtosis: -0.9131490986153108
Entropy: 1.3862943611198906
Energy: 3.7909258908833174e-08
Mean: -4.5320691980461335e-05
Standard deviation: 5.1814015883892795e-05
Skewness: -0.702045226640301
Kurtosis: -1.0378766518022122
Entropy: 2.0794415416798357
Energy: 1.379149883758652e-07
Mean: 2.585784313725488e-06
Standard deviation: 9.585221060504502e-05
Skewness: 0.6048584819978272
Kurtosis: 0.05237886854066387
Entropy: 2.70805020110221
Energy: 1.3353107128267983e-07
Mean: 1.3908166468191294e-05
Standard deviation: 6.525027674206248e-0

In [ ]:
# Measure the time it takes to preprocess the whole edf file